# Test to see what Jhon is talking about

In [2]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [3]:
%load_ext autoreload
%autoreload 2

from gbmhackathon.training.predictive import *
from gbmhackathon.models.mme import MultiModalEncoder, ClinicalLinkageModule, concat_modality_embeddings
from gbmhackathon.utils.module_functions import instantiate
from gbmhackathon.utils.model_saving import *
from gbmhackathon.s3_loader import load_s3

import os, time
from copy import deepcopy
import numpy as np
import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
# To investigate gradients
from torchviz import make_dot

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load embeddings

In [4]:
PATH = './mme_embeddings/MultiModalEncoder_E100_lr1e-03_CosineAnnealingLR_etamin5e-05_t0.05_ntxent_mod5.pkl'
with open(PATH, 'rb') as f:
    mme_embeddings = pkl.load(f)

In [6]:
mme_embeddings['hne'].size()

torch.Size([114, 32])

## Load Dataset

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

In [10]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-05-25_14-36_new_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"
                }
pkl_storage_folder = "embedding_V1"

In [11]:
dataset = PredictiveLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=0.0, normalize=True)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 114 
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_predictive, generator=torch.Generator(device=dataset.device))

cas 2 : device reconnu : cuda 
Using device: cuda
spatial tensor([-1.2553, -1.1431, -1.0777, -1.2676, -0.8982, -1.1545, -0.8487, -1.2577,
        -0.9547, -1.1874, -1.0252, -1.2998, -1.3097, -1.2077, -1.3141, -1.2162,
        -1.0057, -1.2872, -0.9572, -1.0273, -1.0509, -1.0739, -0.9225, -1.1718,
        -1.1085, -1.2474, -1.2076, -1.3897, -1.0923, -1.1740, -1.0682, -1.4825,
        -1.2190, -1.1134, -0.9512, -1.2223, -0.9888, -1.1704, -0.9840, -1.0893,
        -1.2488, -1.1461, -1.0863, -1.0485, -1.1140, -0.9333, -1.0830, -1.0695,
        -1.1005, -0.9239, -1.3218, -0.9554, -1.0725, -1.1427, -1.0236, -1.1748,
        -0.9028, -1.0569, -1.0684, -1.0735, -0.9262, -1.0439, -0.9128, -1.0088,
        -1.0696, -1.1391, -0.8199, -0.8497, -1.4329, -1.0332, -0.8422, -1.0564,
        -1.0540, -0.9929, -0.9593, -1.2920, -1.2158, -1.0403, -0.9243, -1.1929,
        -0.9487, -1.1276, -1.3280, -0.8531, -1.1883, -1.0247], device='cuda:0')
Normalization applied successfully
Dataset size: 114


## We reconstruct the batch all as it is done in all notebooks (this is the way the batch all was constructed to make the inference that is loaded earlier)

In [12]:
# Batch of all true samples (no dropout augmented samples)
batch_all = [dataset.__getitem__(idx) for idx in dataset.ind2patient if 'd' not in dataset.ind2patient[idx]]
batch_all = collate_predictive(batch_all)

## Retrieve Available modalities

In [32]:
avail_mods = batch_all[3]

## Retrieve modalities in this dataset

In [33]:
modality_list = batch_all[1]

## Retrieve lists of patients sample ids

In [34]:
patient_list = batch_all[0]

## Functions to filter rows and only keep the ones that have the modality

In [59]:
def get_modality_keys(row, avail_mods):
    avail_mod_row = avail_mods[row]
    return [mod for i, mod in enumerate(modality_list) if avail_mod_row[i] == 1]

def get_row_id(i, patient_list):
    return patient_list[i]
    
def get_filter(tensor, modality, avail_mods):
    keep_row = []
    for i in range(tensor.size(0)):
        if modality in get_modality_keys(i, avail_mods):
            keep_row.append(i)
    return keep_row

def filter_embeddings(mme_embeddings, modality, avail_mods):
    return mme_embeddings[get_filter(mme_embeddings, modality, avail_mods),:]

def get_filtered_info(mme_emb_dict, avail_mods, patient_list):
    filtered_dict = {}
    filtered_ids_dict = {}
    for key in mme_emb_dict.keys():
        filtered_dict[key] = filter_embeddings(mme_emb_dict[key], key, avail_mods)
        row_filter = get_filter(mme_emb_dict[key], key, avail_mods)
        filtered_ids_dict[key] = [get_row_id(i, patient_list) for i in row_filter]
    return filtered_dict, filtered_ids_dict


In [ ]:
def prepare_embeddings(emb_dict, batch_all):
    patient_list, modality_list, avail_mods = batch_all[0], batch_all[1], batch_all[3]
    return get_filtered_info(mme_embeddings, avail_mods, patient_list)

## Filter embeddings for each modality

In [60]:
filtered_mme_embs, filtered_ids_dict = get_filtered_info(mme_embeddings, avail_mods, patient_list)

In [51]:
filtered_mme_embs["hne"].size()

torch.Size([86, 32])

In [63]:
filtered_mme_embs["hne"]

tensor([[ 1.4452, -0.7095,  1.0359,  ..., -1.1168,  0.5302,  0.5632],
        [ 0.1698, -0.2757, -0.1664,  ..., -0.0247, -0.1694, -0.0324],
        [-0.0501, -1.9580,  0.6747,  ..., -1.0045,  0.6620,  1.1135],
        ...,
        [-0.7881, -1.1238,  0.2047,  ...,  0.4879,  0.6982,  0.0747],
        [ 0.7238,  0.5267,  0.3655,  ...,  0.4581, -0.1684,  0.1523],
        [ 1.0132, -0.2212, -0.0264,  ..., -0.6552, -0.4293,  0.2691]],
       grad_fn=<IndexBackward0>)

In [66]:
filtered_ids_dict['hne']

['HK_G_071a',
 'HK_G_028a',
 'HK_G_043a',
 'HK_G_104a',
 'HK_G_087a',
 'HK_G_095a',
 'HK_G_106a',
 'HK_G_036b',
 'HK_G_015a',
 'HK_G_092b',
 'HK_G_058a',
 'HK_G_107a',
 'HK_G_012a',
 'HK_G_008a',
 'HK_G_029b',
 'HK_G_016a',
 'HK_G_096b',
 'HK_G_040a',
 'HK_G_035a',
 'HK_G_041a',
 'HK_G_002a',
 'HK_G_101a',
 'HK_G_048a',
 'HK_G_027a',
 'HK_G_059b',
 'HK_G_023a',
 'HK_G_056a',
 'HK_G_067a',
 'HK_G_115b',
 'HK_G_004a',
 'HK_G_070a',
 'HK_G_013a',
 'HK_G_057a',
 'HK_G_022a',
 'HK_G_021a',
 'HK_G_066a',
 'HK_G_082b',
 'HK_G_001a',
 'HK_G_026a',
 'HK_G_091a',
 'HK_G_093a',
 'HK_G_039a',
 'HK_G_024a',
 'HK_G_011a',
 'HK_G_050a',
 'HK_G_019a',
 'HK_G_031a',
 'HK_G_111b',
 'HK_G_063a',
 'HK_G_017b',
 'HK_G_114a',
 'HK_G_072a',
 'HK_G_010a',
 'HK_G_049a',
 'HK_G_007a',
 'HK_G_062a',
 'HK_G_037a',
 'HK_G_042a',
 'HK_G_044b',
 'HK_G_080a',
 'HK_G_112a',
 'HK_G_025a',
 'HK_G_033a',
 'HK_G_052a',
 'HK_G_005a',
 'HK_G_046a',
 'HK_G_098b',
 'HK_G_084b',
 'HK_G_047a',
 'HK_G_090b',
 'HK_G_060a',
 'HK_G